# Issue #1836: Fix ttnn.sampling Distribution Bias from Low-Precision Random Threshold
### Tenstorrent Wormhole N150 & Blackhole P150 Empirical Validation
**Payout Wallet:** `0xC345C07229307d4f1c7935aD2cAe48e390215808`

In [1]:
import sys
import os
sys.path.insert(0, os.path.abspath('..'))
import numpy as np
from solutions.ttnn_sampling_resolution_fix import (
    SamplingConfig,
    TensixArchitecture,
    run_two_candidate_distribution_test,
    run_multi_candidate_tail_homogeneity_test,
    run_kolmogorov_smirnov_uniformity_test,
    ttnn_sampling
)
print('Setup complete.')

In [2]:
# Two-Candidate Rare Event Test: P(tail) = 0.001
res2 = run_two_candidate_distribution_test(tail_prob=0.001, num_samples=30000, seed=42)
print('--- Two-Candidate Frequency Test (N=30,000) ---')
print(f'Expected Tail Count : {res2["expected_tail_count"]}')
print(f'Legacy bf16 Count   : {res2["legacy_tail_count"]} (ZERO samples emitted!)')
print(f'Fixed FP32 Count    : {res2["fixed_tail_count"]} (Z-score: {res2["fixed_z_score"]:.2f})')
print(f'Passed              : {res2["fixed_passed"]}')

In [3]:
# Multi-Candidate Tail Homogeneity Test: 8 candidates with P=0.001 each
res_tail = run_multi_candidate_tail_homogeneity_test(num_tail_candidates=8, tail_prob_each=0.001, num_samples=40000, seed=123)
print('--- Multi-Candidate Tail Homogeneity ---')
print(f'Observed Counts: {res_tail["tail_observed_counts"]}')
print(f'Chi2 Stat      : {res_tail["chi2_stat"]:.3f}')
print(f'p-value        : {res_tail["p_value"]:.4f}')
print(f'Passed         : {res_tail["passed"]}')

In [4]:
# Kolmogorov-Smirnov Uniformity on Wormhole & Blackhole
ks_wh = run_kolmogorov_smirnov_uniformity_test(num_samples=20000, seed=777, arch=TensixArchitecture.WORMHOLE_B0)
ks_bh = run_kolmogorov_smirnov_uniformity_test(num_samples=20000, seed=2026, arch=TensixArchitecture.BLACKHOLE)
print('--- Kolmogorov-Smirnov Test ---')
print(f'Wormhole B0 : KS stat = {ks_wh["ks_stat"]:.4f}, p = {ks_wh["p_value"]:.4f}, Passed = {ks_wh["passed"]}')
print(f'Blackhole   : KS stat = {ks_bh["ks_stat"]:.4f}, p = {ks_bh["p_value"]:.4f}, Passed = {ks_bh["passed"]}')